# Bổ sung: Tạo pseudo-label cho ABSA (Keyword + Sentiment)

Mục tiêu của notebook này là **tạo file nhãn ABSA tự động** (pseudo-label) từ dữ liệu review đã crawl ở Notebook 01/02, để Notebook 04 có thể **train ABSA “thực tế”** mà không phụ thuộc vào code ngoài `Notebook_Report/`.

## Input
- `Notebook_Report/absa_clean_reviews.csv` (output từ Notebook 02, đã làm sạch)

## Output (tạo mới trong `Notebook_Report/absa/`)
- `absa_unlabeled.jsonl`: mỗi dòng `{id, tmdb_id, text}`
- `labeled_absa_auto.jsonl`: mỗi dòng thêm `labels: [{aspect, sentiment}, ...]`

## Ý tưởng gán nhãn (pseudo-label)
- **Aspect**: phát hiện bằng keyword theo 6 nhóm: `script, acting, visuals, music, pacing, direction` + luôn có `overall`.
- **Sentiment**: có thể chọn 1 trong các cách **lexicon-based**:
  - `vader`: VADER (khuyến nghị nếu cài được `vaderSentiment`)
  - `textblob`: TextBlob/Pattern (nếu cài `textblob`)
  - `sentiwordnet`: SentiWordNet (NLTK)
  - `fallback`: heuristic đơn giản (khi thiếu thư viện)

Lưu ý: pseudo-label **có nhiễu**, mục tiêu là chứng minh pipeline train/val + metric (micro/macro F1) theo barem.


In [1]:
import json
import re
from pathlib import Path

import pandas as pd

# -----------------------------
# Cấu hình I/O trong Notebook_Report/
# -----------------------------
NOTEBOOK_DIR = Path.cwd()  # kỳ vọng bạn đang chạy notebook trong Notebook_Report/
# fallback: nếu mở notebook ở chỗ khác, vẫn cố gắng tìm đúng folder Notebook_Report
if NOTEBOOK_DIR.name != "Notebook_Report":
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "Notebook_Report" / "cinesense_reviews.csv").exists():
            NOTEBOOK_DIR = p / "Notebook_Report"
            break

ABSA_CLEAN_CSV = NOTEBOOK_DIR / "absa_clean_reviews.csv"
OUT_DIR = NOTEBOOK_DIR / "absa"
OUT_DIR.mkdir(parents=True, exist_ok=True)

UNLABELED_JSONL = OUT_DIR / "absa_unlabeled.jsonl"
LABELED_JSONL = OUT_DIR / "labeled_absa_auto.jsonl"

if not ABSA_CLEAN_CSV.exists():
    raise FileNotFoundError(
        f"Không thấy {ABSA_CLEAN_CSV}. Bạn chạy Notebook 02 để tạo file clean cho ABSA trước nhé."
    )

print("Notebook dir:", NOTEBOOK_DIR)
print("Input:", ABSA_CLEAN_CSV)
print("Output dir:", OUT_DIR)


Notebook dir: /Users/kotori/CineSen/Notebook_Report
Input: /Users/kotori/CineSen/Notebook_Report/absa_clean_reviews.csv
Output dir: /Users/kotori/CineSen/Notebook_Report/absa


In [2]:
# -----------------------------
# 1) Export unlabeled JSONL từ CSV đã clean (Notebook 02)
# -----------------------------
LIMIT_ROWS = None  # bám sát quy mô dữ liệu ~9k review của project

# Không xử lý text ở notebook này nữa: chỉ load dữ liệu đã clean
usecols = ["review_id", "tmdb_id", "cleaned_content"]
df = pd.read_csv(ABSA_CLEAN_CSV, usecols=usecols).fillna("")

# Chuẩn tên cột để các bước sau thống nhất dùng `text`
df = df.rename(columns={"cleaned_content": "text"})

# Bỏ dòng trống sau cleaning
df = df[df["text"].astype(str).str.strip().str.len() > 0]

if LIMIT_ROWS is not None:
    df = df.head(LIMIT_ROWS)

print("Rows for unlabeled:", len(df))

n_written = 0
with UNLABELED_JSONL.open("w", encoding="utf-8") as f:
    for _, r in df.iterrows():
        rec = {
            "id": str(r["review_id"]),
            "tmdb_id": int(r["tmdb_id"]),
            "text": str(r["text"]),
        }
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        n_written += 1

print(f"Wrote unlabeled: {n_written} -> {UNLABELED_JSONL}")


Rows for unlabeled: 9472
Wrote unlabeled: 9472 -> /Users/kotori/CineSen/Notebook_Report/absa/absa_unlabeled.jsonl


In [3]:
# -----------------------------
# 2) Auto-label: keyword detect aspect + sentiment (VADER nếu có)
# -----------------------------
ASPECTS = ["script", "acting", "visuals", "music", "pacing", "direction", "overall"]
SENTIMENTS = ["negative", "neutral", "positive"]

ASPECT_KEYWORDS = {
    "script": [
        "script",
        "story",
        "plot",
        "writing",
        "written",
        "screenplay",
        "narrative",
        "storyline",
        "dialogue",
        "dialog",
    ],
    "acting": [
        "acting",
        "performance",
        "performances",
        "actor",
        "actress",
        "cast",
        "starring",
        "played",
        "portrayal",
        "character",
    ],
    "visuals": [
        "visual",
        "visuals",
        "cgi",
        "cinematography",
        "cinematic",
        "effects",
        "special effects",
        "animation",
        "animated",
        "shot",
        "shots",
    ],
    "music": [
        "music",
        "score",
        "soundtrack",
        "sound track",
        "song",
        "songs",
        "composer",
    ],
    "pacing": [
        "pacing",
        "pace",
        "slow",
        "fast",
        "drag",
        "dragged",
        "rushed",
        "length",
        "long",
        "short",
        "boring",
        "tedious",
        "tight",
        "flow",
    ],
    "direction": [
        "direction",
        "director",
        "directed",
        "filmmaking",
        "film-making",
        "helmed",
        "directorial",
    ],
}


def _normalize_for_keyword(s: str) -> str:
    return re.sub(r"[^a-z\s]", " ", (s or "").lower())


def detect_aspects(text: str) -> set[str]:
    normalized = _normalize_for_keyword(text)
    words = set(normalized.split())
    found = {"overall"}
    for aspect, keywords in ASPECT_KEYWORDS.items():
        for kw in keywords:
            if kw in words or kw in normalized:
                found.add(aspect)
                break
    return found


def _fallback_sentiment(text: str) -> str:
    t = (text or "").lower()
    pos = (
        "great",
        "good",
        "amazing",
        "excellent",
        "love",
        "best",
        "brilliant",
        "stunning",
        "outstanding",
        "positive",
    )
    neg = (
        "bad",
        "terrible",
        "weak",
        "boring",
        "worst",
        "awful",
        "poor",
        "disappointing",
        "negative",
        "do not recommend",
    )
    has_pos = sum(1 for w in pos if w in t)
    has_neg = sum(1 for w in neg if w in t)
    if has_pos > has_neg:
        return "positive"
    if has_neg > has_pos:
        return "negative"
    return "neutral"


# Chọn 1 trong: "vader" | "textblob" | "sentiwordnet" | "fallback"
SENTIMENT_METHOD = "vader"


def _sentiment_from_polarity(p: float, neg_th: float = -0.05, pos_th: float = 0.05) -> str:
    if p <= neg_th:
        return "negative"
    if p >= pos_th:
        return "positive"
    return "neutral"


def _sentiment_vader(text: str) -> str:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

    analyzer = SentimentIntensityAnalyzer()
    compound = analyzer.polarity_scores(text)["compound"]
    return _sentiment_from_polarity(compound)


def _sentiment_textblob(text: str) -> str:
    # TextBlob polarity: [-1, 1]
    from textblob import TextBlob

    polarity = float(TextBlob(text).sentiment.polarity)
    return _sentiment_from_polarity(polarity)


def _sentiment_sentiwordnet(text: str) -> str:
    # Lexicon-based sentiment bằng SentiWordNet.
    # Lưu ý: đây là heuristic (tổng score), không mạnh bằng model.
    import nltk
    from nltk import pos_tag, word_tokenize
    from nltk.corpus import sentiwordnet as swn
    from nltk.corpus import wordnet as wn

    # cố gắng tải resource nếu thiếu
    for pkg in [
        ("punkt", "tokenizers/punkt"),
        ("averaged_perceptron_tagger", "taggers/averaged_perceptron_tagger"),
        ("wordnet", "corpora/wordnet"),
        ("sentiwordnet", "corpora/sentiwordnet"),
    ]:
        name, path = pkg
        try:
            nltk.data.find(path)
        except LookupError:
            nltk.download(name, quiet=True)

    def _to_wn_pos(tag: str):
        if tag.startswith("J"):
            return wn.ADJ
        if tag.startswith("V"):
            return wn.VERB
        if tag.startswith("N"):
            return wn.NOUN
        if tag.startswith("R"):
            return wn.ADV
        return None

    tokens = word_tokenize((text or "").lower())
    tagged = pos_tag(tokens)

    score = 0.0
    hit = 0

    for w, t in tagged:
        wn_pos = _to_wn_pos(t)
        if wn_pos is None:
            continue
        # lấy synset đầu tiên như heuristic
        synsets = wn.synsets(w, pos=wn_pos)
        if not synsets:
            continue
        syn = synsets[0]
        swn_syn = swn.senti_synset(syn.name())
        score += float(swn_syn.pos_score()) - float(swn_syn.neg_score())
        hit += 1

    if hit == 0:
        return "neutral"

    avg = score / hit
    return _sentiment_from_polarity(avg)


def get_sentiment(text: str) -> str:
    method = (SENTIMENT_METHOD or "fallback").strip().lower()

    try:
        if method == "vader":
            return _sentiment_vader(text)
        if method == "textblob":
            return _sentiment_textblob(text)
        if method == "sentiwordnet":
            return _sentiment_sentiwordnet(text)
        return _fallback_sentiment(text)
    except Exception:
        # nếu thiếu thư viện / lỗi runtime -> fallback an toàn
        return _fallback_sentiment(text)


def auto_label_record(rec: dict) -> dict:
    text = (rec.get("text") or "").strip()
    if not text:
        rec["labels"] = []
        return rec

    sentiment = get_sentiment(text)
    if sentiment not in SENTIMENTS:
        sentiment = "neutral"

    aspects = detect_aspects(text)
    rec["labels"] = [{"aspect": a, "sentiment": sentiment} for a in sorted(aspects)]
    return rec


# gán nhãn và ghi ra file labeled
n_labeled = 0
with UNLABELED_JSONL.open("r", encoding="utf-8") as fin, LABELED_JSONL.open("w", encoding="utf-8") as fout:
    for line in fin:
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        auto_label_record(rec)
        fout.write(json.dumps(rec, ensure_ascii=False) + "\n")
        n_labeled += 1

print(f"Wrote labeled: {n_labeled} -> {LABELED_JSONL}")


Wrote labeled: 9472 -> /Users/kotori/CineSen/Notebook_Report/absa/labeled_absa_auto.jsonl


In [4]:
# -----------------------------
# 3) Thống kê nhanh để báo cáo
# -----------------------------
from collections import Counter
from itertools import islice

aspect_counter = Counter()
sent_counter = Counter()

# Thống kê trên toàn bộ file (hoặc giới hạn nếu muốn)
STATS_LIMIT = None  # None = đọc hết; đặt số (vd 2000) nếu muốn xem nhanh

with LABELED_JSONL.open("r", encoding="utf-8") as f:
    it = f if STATS_LIMIT is None else islice(f, STATS_LIMIT)
    for line in it:
        rec = json.loads(line)
        labels = rec.get("labels", [])
        for lab in labels:
            aspect_counter[lab.get("aspect")] += 1
            sent_counter[lab.get("sentiment")] += 1

print("Top aspects:")
for k, v in aspect_counter.most_common(10):
    print(f"  {k}: {v}")

print("\nSentiment counts:")
for k, v in sent_counter.most_common():
    print(f"  {k}: {v}")


Top aspects:
  overall: 9472
  acting: 6138
  script: 5913
  pacing: 4298
  visuals: 3695
  direction: 2094
  music: 1865

Sentiment counts:
  positive: 21447
  neutral: 8649
  negative: 3379
